In [ ]:
# NOTEBOOK NAME
# FeatureStatsSandbox.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd

# # from pathlib import Path      # used to play with pathnames to save

# # from PIL import Image         # used for creating gif loops
# import os                     # used for retrieving file names

# # mapping things
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.io import shapereader

# # for adding lat/lon gridlines on plots
# import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER


# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions/')
from CustomFunctions1 import *
from RadarPlotsCustomFunctions import *
# sys.path.insert(0, '/scratch/v46/sg3241/tmp/PyFLEXTRKR/')  # folder *containing* pyflextrkr/

# import logging
# logging.basicConfig(level=logging.INFO)
# from pyflextrkr.idcells_reflectivity import idcells_reflectivity
# from pyflextrkr.tracksingle_driver import tracksingle_driver
# from pyflextrkr.gettracks import gettracknumbers
# from pyflextrkr.trackstats_driver import trackstats_driver

# # for adding a colourful topo base map to the CAPI plots
# from custom_elevation import fetch_srtm, fetch_gebco_local
# from matplotlib.colors import LinearSegmentedColormap
# from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

# import glob

# import shutil # I think this is to rename the output file from PyFLEXTRKR
# import gc # something to prevent memory leaks

# from pathlib import Path      # used to play with pathnames to save

# from PIL import Image         # used for creating gif loops
# import os                     # used for retrieving file names

# import matplotlib.colors as mcolors
# import matplotlib.cm as cm

# # from matplotlib.patches import Circle # for radar range ring circles on the map
# # from matplotlib.lines import Line2D # for plotting stars on the map legend

In [ ]:
# FEATURE DATA LOADING


# get rid of them BS warnings I don't care about
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='argopy')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='xarray')


RadarIDno              = 22           # [see Radar Number Catalogue below (int)]                   # Which radar site are you plotting data for?
PlotDate               = '2024-03-09' # ['YYYY-MM-DD' (string)]                                    # Which (UTC) date are you plotting for?

# retrieve pieces of the date and construct one without dashes
YearStr = PlotDate[0:4]
MonthStr = PlotDate[5:7]
DayStr = PlotDate[8:10]
FileDateStr = YearStr + MonthStr + DayStr

FeatureStoragePath = f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/{RadarIDno}/V35/{FileDateStr}/{RadarIDno}_{FileDateStr}FeatureStats.nc'
FeatureXR = xr.open_dataset(FeatureStoragePath)

# Create a new time dimension for time of day, instead of just time in a given feature's life
# Generate 288 x 5-minute timestamps to assign as coordinate along the 'times' dimension
RadarFileDatePD = pd.Timestamp(PlotDate).date()
FrameTimes = pd.date_range(start=pd.Timestamp(RadarFileDatePD), periods=288, freq='5min')
FeatureXR = FeatureXR.assign_coords(FrameTimes=xr.DataArray(FrameTimes, dims='FrameTimes'))

# add variables that rearrange times by time of day rather than time in a given feature's life
FeatureXR = AddFrameTimeVars(FeatureXR)

In [ ]:
FeatureXR

In [ ]:
# CHAD PLOT
# SCATTER PLOT BETWEEN TWO CHOSEN VARIABLES WITH COLOUR AS ANOTHER VARIABLE

# ---------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------
XVarName     = 'cell_meanlon_frametimes'        # string name of the x-axis variable
YVarName     = 'cell_meanlat_frametimes'     # string name of the y-axis variable
ColourVarName = 'maxETH_30dbz_frametimes'   # string name of the colour variable, or None for no colouring

ColourMap    = make_ChadMapZ()       # colourmap to use if colouring by a third variable
PlotColourStyle = 'Dark'             # 'Light' or 'Dark'
# ---------------------------------------------------------------

StandOutColour = 'white'
VarName =            '30 dBZEcho Top Height'
VarNameLong =        'corrected_reflectivity'
VarMinVal =          0       # [dBZ]
VarMaxVal =           6.5       # [dBZ]
VarUnit =            'km'
VarFillValue =       -32.0     # [dBZ]
VarColourBar =       make_ChadMapZ()
VarColourBar_min =   -3.0
VarColourBar_max =   10.0
VarColourBar_norm =  Normalize(vmin=-3.0, vmax=10.0)
VarTickSpacing =     1.0



# Pull the two main variables and flatten to 1D
XData = FeatureXR[XVarName].values.flatten()
YData = FeatureXR[YVarName].values.flatten()

# Build a valid-data mask — only keep points where BOTH x and y are finite
ValidMask = np.isfinite(XData) & np.isfinite(YData)

# If a colour variable is given, also require it to be finite
if ColourVarName is not None:
    CData     = FeatureXR[ColourVarName].values.flatten()
    ValidMask = ValidMask & np.isfinite(CData)
    CDataValid = CData[ValidMask]
else:
    CDataValid = None

# Apply the mask
XDataValid = XData[ValidMask]
YDataValid = YData[ValidMask]

print(f'{np.sum(ValidMask)} valid points out of {len(XData)} total')

# ---------------------------------------------------------------
# PLOTTING
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))

if PlotColourStyle == 'Dark':
    fig.patch.set_facecolor('#0a0a0a')
    ax.set_facecolor('#0a0a0a')
    StandOutColour = 'white'
else:
    StandOutColour = 'black'

# Scatter plot
if CDataValid is not None:
    sc = ax.scatter(XDataValid, YDataValid, c=CDataValid, cmap=ColourMap, norm=VarColourBar_norm,
                    s=5, alpha=0.8, linewidths=0)

     # GridViewer = ax.pcolormesh( PlotLons, PlotLats, PlotVar, cmap=VarColourBar, norm=VarColourBar_norm, shading='auto', transform=ccrs.PlateCarree() )

    cbar = fig.colorbar(sc, ax=ax)
    # colour bar controls
    # cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
    cbar.set_label(VarName + ' [' + VarUnit + ']', color = StandOutColour)
    cbar.ax.yaxis.set_tick_params(color = StandOutColour)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color = StandOutColour)
    cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+0.00001, VarTickSpacing))  # control tick spacing
else:
    ax.scatter(XDataValid, YDataValid, s=5, alpha=0.8, linewidths=0,
               color=StandOutColour)

# Labels and styling
ax.set_xlabel(XVarName, color=StandOutColour)
ax.set_ylabel(YVarName, color=StandOutColour)
ax.tick_params(colors=StandOutColour)
for spine in ax.spines.values():
    spine.set_edgecolor(StandOutColour)

ax.set_title(f'{YVarName} vs {XVarName}', color=StandOutColour)

plt.grid()
# plt.xlim([0,500])
# plt.ylim([0,60])

plt.tight_layout()
plt.show()


In [ ]:
# CHAD PLOT
# SCATTER PLOT BETWEEN TWO CHOSEN VARIABLES WITH COLOUR AS ANOTHER VARIABLE

# ---------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------
XVarName     = 'maxETH_20dbz_frametimes'       # string name of the x-axis variable
YVarName     = 'maxETH_30dbz_frametimes'     # string name of the y-axis variable
ColourVarName = 'cell_meanlon_frametimes'  # string name of the colour variable, or None for no colouring

ColourMap    = make_ChadMapZ()       # colourmap to use if colouring by a third variable
PlotColourStyle = 'Dark'             # 'Light' or 'Dark'
# ---------------------------------------------------------------

StandOutColour = 'white'
VarName =            '30 dBZEcho Top Height'
VarNameLong =        'corrected_reflectivity'
VarMinVal =           148      # [dBZ]
VarMaxVal =           150       # [dBZ]
VarUnit =            'km'
VarFillValue =       -32.0     # [dBZ]
VarColourBar =       make_ChadMapZ()
VarColourBar_min =   147
VarColourBar_max =   153
VarColourBar_norm =  Normalize(vmin=147, vmax=153)
VarTickSpacing =     0.5



# Pull the two main variables and flatten to 1D
XData = FeatureXR[XVarName].values.flatten()
YData = FeatureXR[YVarName].values.flatten()

# Build a valid-data mask — only keep points where BOTH x and y are finite
ValidMask = np.isfinite(XData) & np.isfinite(YData)

# If a colour variable is given, also require it to be finite
if ColourVarName is not None:
    CData     = FeatureXR[ColourVarName].values.flatten()
    ValidMask = ValidMask & np.isfinite(CData)
    CDataValid = CData[ValidMask]
else:
    CDataValid = None

# Apply the mask
XDataValid = XData[ValidMask]
YDataValid = YData[ValidMask]

print(f'{np.sum(ValidMask)} valid points out of {len(XData)} total')

# ---------------------------------------------------------------
# PLOTTING
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))

if PlotColourStyle == 'Dark':
    fig.patch.set_facecolor('#0a0a0a')
    ax.set_facecolor('#0a0a0a')
    StandOutColour = 'white'
else:
    StandOutColour = 'black'

# Scatter plot
if CDataValid is not None:
    sc = ax.scatter(XDataValid, YDataValid, c=CDataValid, cmap=ColourMap, norm=VarColourBar_norm,
                    s=5, alpha=0.8, linewidths=0)

     # GridViewer = ax.pcolormesh( PlotLons, PlotLats, PlotVar, cmap=VarColourBar, norm=VarColourBar_norm, shading='auto', transform=ccrs.PlateCarree() )

    cbar = fig.colorbar(sc, ax=ax)
    # colour bar controls
    # cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
    cbar.set_label(VarName + ' [' + VarUnit + ']', color = StandOutColour)
    cbar.ax.yaxis.set_tick_params(color = StandOutColour)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color = StandOutColour)
    cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+0.00001, VarTickSpacing))  # control tick spacing
else:
    ax.scatter(XDataValid, YDataValid, s=5, alpha=0.8, linewidths=0,
               color=StandOutColour)

# Labels and styling
ax.set_xlabel(XVarName, color=StandOutColour)
ax.set_ylabel(YVarName, color=StandOutColour)
ax.tick_params(colors=StandOutColour)
for spine in ax.spines.values():
    spine.set_edgecolor(StandOutColour)

ax.set_title(f'{YVarName} vs {XVarName}', color=StandOutColour)

plt.grid()
# plt.xlim([0,500])
# plt.ylim([0,60])

plt.tight_layout()
plt.show()

In [ ]:
# --- Load PyFLEXTRKR cell identification file for this timestep ---

RadarIDno = 41
RadarFileTime = '120000'
RadarFileTimePrint = '12:00'
FileDateStr = '20240309'

CellIDPath = (
    '/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/SingleTimes/'
    + str(RadarIDno) + '/V5/' + FileDateStr + '/'
    + f'cellidfile_{FileDateStr}_{RadarFileTime}.nc'
)
if os.path.exists(CellIDPath):
    ds_cellid = xr.open_dataset(CellIDPath)
    HasCellID = True
else:
    print(f'No cell ID file found for {RadarFileTimePrint}, skipping Steiner overlay')
    HasCellID = False

In [ ]:
ds_cellid

In [ ]:
 # create the figure to be plotted on with a map projection
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# set the plot background to black and text to white if the user wants it
if (PlotColourStyle=='Dark'):
    fig.patch.set_facecolor('#0a0a0a')  
    ax.set_facecolor('#0a0a0a')
    StandOutColour = 'white'
elif (PlotColourStyle=='Light'):
    StandOutColour = 'black'
else:
    print("PlotColourStyle must be exactly 'Light' or 'Dark'")

# add a background topo map to the plot
ElevationModelPath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
# Extract plot bounds from radar grid
# LonMin, LonMax = float(RadarXR.lon.min()) + LonShift, float(RadarXR.lon.max()) + LonShift
# LatMin, LatMax = float(RadarXR.lat.min()), float(RadarXR.lat.max())

LonMin, LonMax = 145, 150
LatMin, LatMax = -24, -19

# call the "adding-the-map" function
AddTopoShading(ax, LonMin, LonMax, LatMin, LatMax, ElevationModelPath, PlotColourStyle)


# PLOT THE MAIN VARIABLE
# if (DataSourceType == 'Grid'):
    # # index in the netcdf altitude variable for the altitude you want
    # alti = np.where(RadarXR.z == Altitude) # this is a double nested array for some reason
    # alti = alti[0][0] # take the index out of the double nested array

    # # quit out if the altitude does not correspond to one in the netCDF file
    # if ( np.size(alti) != 1): 
    #     raise ValueError( str(Altitude) + ' m is not a valid altitude in the data')
    
    # PlottingArray = RadarXR[VarNameLong][0,alti,:,:]
    # GridViewer = ax.pcolormesh(RadarXR.lon+ LonShift, RadarXR.lat, PlottingArray, 
    #                            cmap=VarColourBar , norm=VarColourBar_norm, transform=ccrs.PlateCarree())

    # # plot a star for the location of the radar on the map
    # ax.plot(float(RadarXR.radar_longitude[0]) + LonShift, float(RadarXR.radar_latitude[0]),
    #         marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=20)
    # ax.plot(float(RadarXR.radar_longitude[0]) + LonShift, float(RadarXR.radar_latitude[0]),
    #         marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=21)

    # plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nat ' + str(Altitude) + ' m Altitude\non ' + \
    #     PlotDate + ' at ' + \
    #     str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC', color = StandOutColour)

# # colour bar controls
# cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
# cbar.set_label(VarName + ' [' + VarUnit + ']', color = StandOutColour)
# cbar.ax.yaxis.set_tick_params(color = StandOutColour)
# plt.setp(cbar.ax.yaxis.get_ticklabels(), color = StandOutColour)
# cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
# cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+0.00001, VarTickSpacing))  # control tick spacing

# ADD LAT/LON GRID LINES 
# (set thickness of grid lines) 
ThinLineThickness    = 0.2
MediumLineThickness  = 0.3
ThickLineThickness   = 0.6
# (and what multiples of lat/lon have lines)
ThinLineFrequency    = 0.1
MediumLineFrequency  = 0.5
ThickLineFrequency   = 1.0
# alright, lets add those lines
AddGridlines(ax, ThinLineThickness, MediumLineThickness, ThickLineThickness,
                 ThinLineFrequency,  MediumLineFrequency,  ThickLineFrequency, StandOutColour)

# set the plot limits to the same as the BACKGROUND TOPO MAP
plt.xlim([LonMin, LonMax])
plt.ylim([LatMin, LatMax])

